# Add SpatialData

- Convert the Cohort of Fobs and labels to a single `SpatialData` object. 
- Save the `SpatialData` object to `LaminDB`.


## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Importing Libraries

In [ ]:
from upath import UPath
import buckaroo  # noqa: F401
import pandas as pd
import natsort as ns
import lamindb as ln
from nbl.util import DaskLocalCluster, reset_table_index
import nbl

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [ ]:
ln.context.uid = "FGEcC5bGULbo0000"
ln.context.version = "1"
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"

ln.context.track()

In [ ]:
ln.setup.settings.instance

In [ ]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

## Convert FOVs to SpatialData

### Set up Paths

In [ ]:
original_data_path = UPath("../../../data/raw/original_data/nbl_cohort")
fov_dir: UPath = original_data_path / "images"
label_dir: UPath = original_data_path / "segmentation" / "labels"

In [ ]:
hu_data_path: UPath = ln.settings.storage.root / "Hu.zarr"
nbl_data_path: UPath = ln.settings.storage.root / "nbl.zarr"

### Convert Control Cohort to SpatialData - Hu Data

#### Convert Cohort

In [ ]:
hu_sdata = nbl.io.convert_cohort(
    fov_dir=fov_dir, label_dir=label_dir, filter_fovs=r"Hu-*", file_path=hu_data_path, return_sdata=True
)

#### Aggregate Images by Labels and Compute Region Properties

In [ ]:
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell")
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="nuclear", table_name="nuclear")

In [ ]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [ ]:
nbl.tl.regionprops(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)
nbl.tl.regionprops(sdata=hu_sdata, label_type="nuclear", table_name="nuclear", properties=properties)

### Convert Sample Cohort to SpatialData - NBL Data

#### Convert Cohort

In [ ]:
nbl_sdata = nbl.io.convert_cohort(
    fov_dir=fov_dir, filter_fovs=r"NBL-\d+-R\d+C\d+", label_dir=label_dir, file_path=nbl_data_path, return_sdata=True
)

#### Aggregate Images by Labels and Compute Region Properties

In [ ]:
nbl.tl.aggregate_images_by_labels(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell")

In [ ]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [ ]:
nbl.tl.regionprops(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)

## Add Pixie Clusters to NBL SpatialData

In [ ]:
pixie_clusters_path: UPath = (
    original_data_path / "segmentation" / "cell_table" / "cell_table_size_normalized_cell_labels_noCD117.csv"
)

In [ ]:
pixie_clusters_df = pd.read_csv(pixie_clusters_path).astype({"label": int, "cell_meta_cluster": "category"})

In [ ]:
all_fovs = ns.natsorted(nbl_sdata.coordinate_systems)

In [ ]:
def get_pixie_clusters(df, fovs: str, suffix="whole_cell") -> pd.DataFrame:
    """Gets pixie clusters from the two clustering csv files.

    Parameters
    ----------
    df : pd.DataFrame
        The Pixie cluster DataFrame
    fov : str
        The FOV to subset on

    Returns
    -------
    pd.DataFrame
        A dataframe containing the two clusters and a column for the segmentation label.
    """
    out_df = []
    for fov in fovs:
        fov_rncm = fov.split("-")[-1].split("_")[0]
        no_cd117_pixie: pd.DataFrame = df[df["fov"].str.split("_").map(lambda x: x[-1]) == fov_rncm]
        result_df = (
            no_cd117_pixie.assign(region=f"{fov}_{suffix}", fov=fov)
            .rename(columns={"label": "instance_id", "cell_meta_cluster": "pixie_cluster"})
            .astype(dtype={"instance_id": int, "region": "category", "pixie_cluster": "category"})
            .filter(items=["instance_id", "region", "fov", "pixie_cluster"])
        )
        out_df.append(result_df)

    return pd.concat(out_df)

In [ ]:
pixie_df = pixie_clusters_df.pipe(func=get_pixie_clusters, fovs=all_fovs, suffix="whole_cell")

In [ ]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"]
    .obs.merge(
        right=pixie_df,
        right_on=["instance_id", "region"],
        left_on=["instance_id", "region"],
    )
    .pipe(reset_table_index)
)

## Add Clinical Information

### Load Clinical Data from LaminDB

In [ ]:
clinical_data: pd.DataFrame = ln.Artifact.filter(key__contains="clinical_data").one().load()

In [ ]:
clinical_data

In [ ]:
cols_to_keep = [
    "fov",
    "Risk",
    "Classification",
    "Sex",
    "Ethnicity",
    "Tissue",
]

In [ ]:
filtered_clincial_data = clinical_data.filter(items=cols_to_keep)

In [ ]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"].obs.merge(right=filtered_clincial_data, on="fov").pipe(reset_table_index)
)
nbl_sdata.tables["whole_cell"].strings_to_categoricals()

In [ ]:
nbl.util.write_elements(sdata=nbl_sdata, elements={"tables": ["whole_cell"]})

### `Arcsinh` Transform the NBL Whole Cell Table

In [ ]:
nbl.pp.arcsinh_transform(
    sdata=nbl_sdata,
    table_names="whole_cell",
    shift_factor=0,
    scale_factor=150,
    method="new table",
    write=True,
    inplace=True,
)

In [ ]:
hu_artifact = ln.Artifact(
    data=hu_data_path,
    type="dataset",
    key="Hu.zarr",
    description="Control Tissue",
    revises=ln.Artifact.filter(key__contains="Hu.zarr").one(),
)

hu_artifact.save(upload=True)

In [ ]:
nbl_artifact = ln.Artifact(
    data=nbl_data_path,
    type="dataset",
    key="nbl.zarr",
    description="NBL Tissue Samples",
    revises=ln.Artifact.filter(key__contains="nbl.zarr").one(),
)

nbl_artifact.save(upload=True)

In [ ]:
ln.finish()